# 01 - Data Collection (World Bank WDI) for ALI (2022)

This notebook downloads World Bank indicator data for the Affordable Liveability Index (ALI) for year = 2022, creates an initial country-level dataset and outputs:

- `data/raw/wdi_2022_initial.csv`
- `outputs/figures/missingness_by_indicator.png`

Next notebook will handle cleaning rules and imputation.

In [ ]:
import pandas as pd
import numpy as np
import requests
from functools import reduce
import matplotlib.pyplot as plt

In [ ]:
YEAR = 2022

# Initial indicator set
# World Bank indicator codes: https://data.worldbank.org/indicator
INDICATORS = {
    # Economic Opportunity (EO)
    "NY.GDP.PCAP.PP.KD": "gdp_per_capita_ppp_const",   # GDP per capita, PPP (constant international $)
    "SL.UEM.TOTL.ZS": "unemployment_rate",            # Unemployment, total (% of labor force)
    "SL.TLF.CACT.ZS": "labor_force_participation",    # Labor force participation rate (% ages 15+)

    # Quality of Life & Environment (QLE)
    "SP.DYN.LE00.IN": "life_expectancy",              # Life expectancy at birth, total (years)
    "EN.ATM.PM25.MC.M3": "pm25_exposure",             # PM2.5 air pollution, mean annual exposure (micrograms per cubic meter)
    "VC.IHR.PSRC.P5": "homicide_rate",                # Intentional homicides (per 100,000 people)

    # Cost pressure proxy (CP)
    "PA.NUS.PPP": "ppp_conversion_factor"             # PPP conversion factor (LCU per international $)
}

In [ ]:
def fetch_world_bank_indicator(ind_code: str, year: int) -> pd.DataFrame:
    """
    Fetch one World Bank indicator for all countries for given year.
    Returns a normalized DataFrame of API records.
    """
    url = (
        "https://api.worldbank.org/v2/country/all/indicator/"
        f"{ind_code}?format=json&per_page=20000&date={year}"
    )
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    payload = r.json()

    # Expected: [metadata, records]
    if not isinstance(payload, list) or len(payload) < 2:
        raise ValueError(f"Unexpected response format for {ind_code}: {type(payload)}")

    records = payload[1] or []
    df = pd.json_normalize(records)
    return df

In [ ]:
frames = []
failed = []

for code, short in INDICATORS.items():
    try:
        df = fetch_world_bank_indicator(code, YEAR)

        if df.empty:
            failed.append((code, "empty dataframe"))
            continue

        required_cols = ["country.value", "countryiso3code", "date", "value"]
        missing_cols = [c for c in required_cols if c not in df.columns]
        if missing_cols:
            failed.append((code, f"missing cols: {missing_cols}"))
            continue

        df = df[required_cols].copy()
        df.rename(
            columns={
                "country.value": "country",
                "countryiso3code": "iso3",
                "date": "year",
                "value": short,
            },
            inplace=True,
        )

        frames.append(df)
        print(f"OK: {code} -> {short}, rows={len(df)}")

    except Exception as e:
        failed.append((code, str(e)))
        print(f"FAIL: {code} ({short}) -> {e}")

print("\nSummary")
print("Indicators requested:", len(INDICATORS))
print("Indicators pulled:", len(frames))
print("Failed indicators:", failed)

In [ ]:
if len(frames) == 0:
    raise RuntimeError("No indicators were downloaded. Check your internet connection or indicator codes.")

data = reduce(
    lambda left, right: pd.merge(left, right, on=["iso3", "year", "country"], how="outer"),
    frames
)

data["year"] = data["year"].astype(int)
data.head(), data.shape

In [ ]:
# Drop entries without ISO3 (usually aggregates)
data = data.dropna(subset=["iso3"]).copy()

# Drop common World Bank aggregate regions/income groups
AGGREGATE_ISO3 = set([
    "WLD","HIC","MIC","LIC",
    "EAP","ECA","ECS","EMU","EUU",
    "LAC","LCN","MNA","MEA","NAC",
    "OED","SAS","SSA"
])

data = data[~data["iso3"].isin(AGGREGATE_ISO3)].copy()

# Keep only ISO3 codes that look valid
data = data[data["iso3"].str.fullmatch(r"[A-Z]{3}", na=False)].copy()

data.shape

In [ ]:
missing = data.isna().mean().sort_values(ascending=False)
missing

In [ ]:
missing_ind = missing.drop(["country", "iso3", "year"], errors="ignore").sort_values(ascending=False)

plt.figure(figsize=(10, 5))
missing_ind.plot(kind="bar")
plt.title("Missingness by indicator (share missing)")
plt.ylabel("Share missing")
plt.tight_layout()

fig_path = "../outputs/figures/missingness_by_indicator.png"
plt.savefig(fig_path, dpi=200)
fig_path

In [ ]:
out_path = "../data/raw/wdi_2022_initial.csv"
data.to_csv(out_path, index=False)
out_path

In [ ]:
data.sample(5, random_state=42)